RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [2]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [3]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data/pdf")

Found 5 PDF files to process

Processing: 1-s2.0-S2665917423000594-main.pdf
  ✓ Loaded 6 pages

Processing: Bone_Fracture_Classification_in_X-Ray_Using_Deep_Learning_Models.pdf
  ✓ Loaded 10 pages

Processing: c8749a1bbb5599be4b2b32673ee135c04efc.pdf
  ✓ Loaded 9 pages

Processing: dae4227fc2337c93b4d949598ced1d3e.pdf
  ✓ Loaded 8 pages

Processing: example1.pdf
  ✓ Loaded 6 pages

Total documents loaded: 39


In [30]:

all_pdf_documents


[Document(metadata={'producer': 'Acrobat Distiller 8.1.0 (Windows)', 'creator': 'Elsevier', 'creationdate': '2023-06-22T05:55:57+00:00', 'crossmarkdomains[1]': 'elsevier.com', 'crossmarkmajorversiondate': '2010-04-23', 'creationdate--text': '22nd June 2023', 'elsevierwebpdfspecifications': '7.0', 'robots': 'noindex', 'moddate': '2023-06-22T07:18:54+00:00', 'author': 'Kosrat Dlshad Ahmed', 'doi': '10.1016/j.measen.2023.100723', 'keywords': 'X-ray,Image fracture,Machine learning,Image processing. classification', 'title': 'Detection of bone fracture based on machine learning techniques', 'subject': 'Measurement: Sensors, 27 (2023) 100723. doi:10.1016/j.measen.2023.100723', 'crossmarkdomains[2]': 'sciencedirect.com', 'crossmarkdomainexclusive': 'true', 'source': '..\\data\\pdf\\1-s2.0-S2665917423000594-main.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1', 'source_file': '1-s2.0-S2665917423000594-main.pdf', 'file_type': 'pdf'}, page_content='Measurement: Sensors 27 (2023) 100723\nAvai

Embedding And vectorStoreDB

In [31]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [32]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10300.75it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\manna\AppData\Local\Temp\ipykernel_5292\2964522620.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


VectorStore

In [33]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore


Vector store initialized. Collection: pdf_documents
Existing documents in collection: 195


In [34]:
## convert the text to embeddings
texts=[doc.page_content for doc in all_pdf_documents]
texts

['Measurement: Sensors 27 (2023) 100723\nAvailable online 25 February 2023\n2665-9174/© 2023 The Author(s). Published by Elsevier Ltd. This is an open access article under the CC BY-NC-ND license ( http://creativecommons.org/licenses/by-\nnc-nd/4.0/).\nDetection of bone fracture based on machine learning techniques \nKosrat Dlshad Ahmed\n*\n, Roojwan Hawezi \nInformation System Engineering Department, Erbil Polytechnic University, Erbil, Iraq   \nARTICLE INFO  \nKeywords: \nX-ray \nImage fracture \nMachine learning \nImage processing. classification \nABSTRACT  \nComputers have been shown to be valuable in every facet of human life, from banking and online shopping to \ncommunication, education, research and development, and even medical. To help doctors and hospitals better \ncare for their patients, a lot of innovative technical resources have been developed. Because the typical scanner \nfor X-rays produces a fuzzy picture of the bone component in issue, surgeons risk making an inac

In [35]:
## convert the text to embeddings
texts=[doc.page_content for doc in all_pdf_documents]

## Generate embeddings for the documents
embeddings=embedding_manager.generate_embeddings(texts)

## store the documents and embeddings in the vector store
vectorstore.add_documents(all_pdf_documents, embeddings)

Generating embeddings for 39 texts...


Batches: 100%|██████████| 2/2 [00:00<00:00,  3.17it/s]

Generated embeddings with shape: (39, 384)
Adding 39 documents to vector store...
Successfully added 39 documents to vector store
Total documents in collection: 234
